In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
class DigitCNN(nn.Module):
    def __init__(self):
        super(DigitCNN,self).__init__()
        self.conv1=nn.Conv2d(1,16,5)
        self.pool=nn.MaxPool2d(2,2)
        self.conv2=nn.Conv2d(16,32,5)
        self.fc1=nn.Linear(512,120)
        self.fc2=nn.Linear(120,10)
    def forward(self,x):
        x=self.pool(F.relu(self.conv1(x)))
        x=self.pool(F.relu(self.conv2(x)))
        x=x.view(-1,512)
        x=F.relu(self.fc1(x))
        return self.fc2(x)
model=DigitCNN()
print(model)

DigitCNN(
  (conv1): Conv2d(1, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=512, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=10, bias=True)
)


In [4]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [5]:
transform=transforms.Compose([
    transforms.ToTensor(),
])
train_data=torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)
train_loader=DataLoader(train_data,batch_size=64,shuffle=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 49.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.69MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.37MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.21MB/s]


In [6]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer=optimizer=optim.Adam(model.parameters(),lr=0.001)
epochs=3
for epoch in range(epochs):
    running_loss=0.0
    for images,labels in train_loader:
        optimizer.zero_grad()
        outputs=model(images)
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        running_loss+=loss.item()
    print(f"Epochs [{epoch+1}/{epochs}],Loss: {running_loss/len(train_loader):.4f}")

Epochs [1/3],Loss: 0.2226
Epochs [2/3],Loss: 0.0636
Epochs [3/3],Loss: 0.0439


In [7]:
# 1. Load Test Data
test_data = torchvision.datasets.MNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# 2. Set model to evaluation mode (turns off dropout/batchnorm updates)
model.eval()

correct = 0
total = 0

# 3. No gradients needed for testing!
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        
        # Get the class with the highest probability
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy on 10,000 test images: {100 * correct / total:.2f}%')

Accuracy on 10,000 test images: 98.94%
